In [ ]:
# PolyPhen-2 Connected code

In [ ]:
# The PolyPhen-2 calculation was performed using Rhapsody, available at: http://rhapsody.csb.pitt.edu/ with documentation here: https://github.com/prody/rhapsody
# After dowlanding results from server for each protein, from each ZIP file, we took out the predictions .txt file and save it under the protein name. 

In [ ]:
#From dowloanded results, we will continue working just with rhapsody-predictions.txt file for each protein. 
import pandas as pd
import numpy as np
import os

input_folder = r"../GLUT project documentation/Results from Rhapsody/protein name rhapsody-predictions.txt"
output_folder = r"../GLUT project documentation/Results from Rhapsody/Rhapsody_patogenicity Excel"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

# Function for processing a single file
def process_file(filename, output_excel):
    columns = [
        'ID', 'Position', 'AA_ref', 'AA_alt', 'Training', 'Score', 'Prob', 'Class',
        'PolyPhen_score', 'PolyPhen_class', 'EVmutation_score', 'EVmutation_class'
    ]

    df = pd.read_csv(filename, delim_whitespace=True, names=columns, comment='#', na_values='nan')

    df = df[['Position', 'AA_ref', 'PolyPhen_score']].copy()
    df = df.dropna(subset=['PolyPhen_score'])

    result = df.groupby(['Position', 'AA_ref']).agg({'PolyPhen_score': 'mean'}).reset_index()
    result['PolyPhen_score'] = result['PolyPhen_score'].round(3)

    result.to_excel(output_excel, index=False)

# Processing all .txt files
for file in os.listdir(input_folder):
    if file.endswith('.txt'):
        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(output_folder, f'{os.path.splitext(file)[0]}.xlsx')
        process_file(input_path, output_path)
        print(f'Processed file: {file}')
# This can handle just one protein in a row, need to be repeated 14 times for each protein.

In [ ]:
# Counting averages of pathogenicity for the whole protein one by one.
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import re
# Path to the folder with processed files from previous step
input_folder = r"../GLUT project documentation/Results from Rhapsody/Rhapsody_patogenicity Excel"

# Dictionary for storing averages
protein_averages = {}

# Processing every file in the folder
for file in os.listdir(output_folder):
    if file.endswith('.xlsx'):
        filepath = os.path.join(output_folder, file)
        df = pd.read_excel(filepath)

        # Calculating the average if the column exists
        if 'PolyPhen_score' in df.columns:
            avg = round(df['PolyPhen_score'].mean(), 2)
            protein_averages[os.path.splitext(file)[0]] = avg
            
# Function for extracting a number from a string
def extract_number(name):
    match = re.search(r'\b(\d+)\b', name)
    if match:
        return int(match.group(1))
    else:
        return 9999  # large number, if the number is not found (for sorting at the end)
# In each file will be newly counted average of the pathogenicity for the protein. 
# These numbers we rewrite to the .txt file named /pathogenicity_PolyPhen-2.txt/

In [ ]:
# In the next step, to determine the pathogenicity of the protein (intracellular/extracellular domain and transmembrane part), we will use the sorting from the "AlphaMissense connected code GLUT" code.
# We will use the folder with the saved .xlsx files as follows:
import pandas as pd
# Loading files
df = pd.read_excel("../GLUT project documentation/Results from DeepTMHMM/Excel output from sequences/protein name_AMK_output_MIO.xlsx")
df_scores = pd.read_excel("../GLUT project documentation/Results from Rhapsody/Rhapsody_patogenicity Excel/protein name.xlsx")

# Removing spaces in column names
df.columns = [col.strip() for col in df.columns]

# Converting positions to the whole numbers
for col in ['O_position', 'M_position', 'I_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Transfer of positions in the second file
df_scores["Position"] = pd.to_numeric(df_scores["Position"], errors='coerce').astype('Int64')

# Creating a position map -> PolyPhen_score
score_map = df_scores.set_index("Position")["PolyPhen_score"].to_dict()

# Assigning scores
df["O_patogenicity"] = df["O_position"].map(score_map)
df["M_patogenicity"] = df["M_position"].map(score_map)
df["I_patogenicity"] = df["I_position"].map(score_map)

# Displaying the result
print(df.head())

# Saving the result
df.to_excel("output_with_polyphen_MIO_protein name.xlsx", index=False)

In [ ]:
# The pathogenicity averages for individual protein parts were calculated in a newly created .xlsx files: /output_with_pathogenicity_PolyPhen-2MIO_"protein name".xlsx/ using the AVERAGE function. 
# The calculated average was copied to a .txt file using the following code:
import pandas as pd
import os

# Set the path to the folder with the files
folder_path = "../GLUT project documentation/Results from DeepTMHMM/PolyPhen-2"  # A folder containing all newly created files with calculated averages

# Initialization of the list for results
results = []

# Going through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        
        # Find a row with the value 'AVERAGE' in any column
        avg_row = df[df.apply(lambda row: row.astype(str).str.contains("AVERAGE", case=False).any(), axis=1)]
        
        if not avg_row.empty:
            row = avg_row.iloc[0]
            values = [
                row.get("O_patogenicity", None),
                row.get("M_patogenicity", None),
                row.get("I_patogenicity", None)
            ]
            results.append([file, *values])

# Conversion to DataFrame
result_df = pd.DataFrame(results, columns=["filename", "O_patogenicity", "M_patogenicity", "I_patogenicity"])

# Save to text file
result_df.to_csv("PolyPhen-2(MIO).txt", sep="\t", index=False)

# Showing the output
print(result_df.head())

In [ ]:
# We will use a similar procedure for lining residues, binding places, and lining residues without binding places, which were calculated and stored using the "AlphaMissense connected code" code.

In [ ]:
import pandas as pd
# Loading files
df = pd.read_excel("../GLUT project documentation/Binding places and lining residues/Excel files/protein name analysis lr_bp_lr-bp.xlsx")
df_scores = pd.read_excel("../GLUT project documentation/Results from Rhapsody/Rhapsody_patogenicity Excel/protein name.xlsx")

# Removing spaces in column names
df.columns = [col.strip() for col in df.columns]

# Converting positions to the whole numbers
for col in ['lr_position', 'bp_position', 'lr-bp_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# PConverting positions in the second file
df_scores["Position"] = pd.to_numeric(df_scores["Position"], errors='coerce').astype('Int64')

# Creating a position map -> PolyPhen_score
score_map = df_scores.set_index("Position")["PolyPhen_score"].to_dict()

# Assigning scores
df["lr_patogenicity"] = df["lr_position"].map(score_map)
df["bp_patogenicity"] = df["bp_position"].map(score_map)
df["lr-bp_patogenicity"] = df["lr-bp_position"].map(score_map)

# Displaying the result
print(df.head())

# Saving the result
df.to_excel("output_with_polyphen_protein name.xlsx", index=False)


In [ ]:
#The pathogenicity averages for individual protein parts were calculated in a newly created .xlsx file: /output_with_pathogenicity_PolyPhen-2_"protein name".xlsx/ using the AVERAGE function. 
#The calculated average was copied to a .txt file using the following code:
import pandas as pd
import os

# Set the path to the folder with the files
folder_path = "../GLUT project documentation/Binding places and lining residues/PolyPhen-2"  # A folder containing all newly created files with calculated averages

# Initialization of the list for results
results = []

# Going through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        
        # Find a row with the value 'AVERAGE' in any column
        avg_row = df[df.apply(lambda row: row.astype(str).str.contains("AVERAGE", case=False).any(), axis=1)]
        
        if not avg_row.empty:
            row = avg_row.iloc[0]
            values = [
                row.get("lr_patogenicity", None),
                row.get("bp_patogenicity", None),
                row.get("lr-bp_patogenicity", None)
            ]
            results.append([file, *values])

# Conversion to DataFrame
result_df = pd.DataFrame(results, columns=["filename", "lr_patogenicity", "bp_patogenicity", "lr-bp_patogenicity"])

# Save to text file
result_df.to_csv("PolyPhen-2(lr,bp,lr-bp).txt", sep="\t", index=False)

# Showing the output
print(result_df.head())

In [ ]:
#Merging all three newly created .txt files together
import pandas as pd
# Loading three files
df1 = pd.read_csv("../GLUT project documentation/Results from Rhapsody/PolyPhen-2 patogenicity.txt", sep="\t")  
df2 = pd.read_csv("../GLUT project documentation/Results from DeepTMHMM/PolyPhen-2/PolyPhen-2(MIO).txt", sep="\t")
df3 = pd.read_csv("../GLUT project documentation/Binding places and lining residues/PolyPhen-2/PolyPhen-2GLUT(lr,bp,lr-bp).txt", sep="\t")

# Merge data according to the common column 'filename'. This column contains the names of proteins.
merged_df = df1.merge(df2, on="protein", how="outer")
merged_df = merged_df.merge(df3, on="protein", how="outer")

# Saving the resulting file
merged_df.to_csv("connected_file_PolyPhen-2.txt", sep="\t", index=False)


In [ ]:
#Creating the heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Loading the file
df = pd.read_csv("../GLUT project documentation/Conected results/connected_file_PolyPhen-2.txt", sep="\t")

# Setting 'filename' as index
df.set_index("protein", inplace=True)

# Creating a heatmap 
plt.figure(figsize=(10, len(df) * 0.4))
sns.heatmap(
    df,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='gray',
    annot=True,
    fmt=".3f"  # Formátovanie na 2 desatinné miesta
)


plt.title("GLUTs pathogenicity profile PolyPhen-2", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


plt.savefig("heatmap_patogenicityPolyPhen-2.png", dpi=300, bbox_inches='tight')

plt.show()